In [ ]:
import mlflow
import mlflow.catboost
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import geopandas as gpd

# 1. Configurar el nombre del experimento en MLflow (dentro de Databricks)
mlflow.set_experiment("/Users/tu_usuario/bogota_inmobiliario_catboost")

with mlflow.start_run(run_name="CatBoost_Barrios_Nativos"):
    
    print("--- 1. CARGA Y PREPARACIÓN DE DATOS ---")
    matriz = "../../data/matriz_barrios_unificada.csv"
    df = pd.read_csv(matriz)
    
    # Separar datos supervisados vs inferencia
    df_supervised = df[df['v_ref_2026'].notnull()].copy()
    df_inference = df[df['v_ref_2026'].isnull()].copy()

    # Preparar Matriz X e y (conservando barrio y localidad)
    y_sup = df_supervised['v_ref_2026']
    cols_a_excluir = ['geometry', 'v_ref_2015', 'v_ref_2026']
    X_sup = df_supervised.drop(columns=[col for col in cols_a_excluir if col in df_supervised.columns])

    cat_features = ['localidad', 'barrio']
    for col in cat_features:
        if col in X_sup.columns:
            X_sup[col] = X_sup[col].astype(str)

    # Train / Test Split
    X_train, X_test, y_train, y_test = train_test_split(X_sup, y_sup, test_size=0.2, random_state=42)

    # 2. Definición de Hiperparámetros y registro en MLflow
    params = {
        "iterations": 600,
        "learning_rate": 0.04,
        "depth": 6,
        "loss_function": "RMSE",
        "random_seed": 42
    }
    mlflow.log_params(params)

    # Entrenamiento con soporte nativo de categorías
    cat_model = CatBoostRegressor(**params, verbose=100)
    cat_model.fit(X_train, y_train, cat_features=cat_features)

    # Predicción y Evaluación
    y_pred_test = cat_model.predict(X_test)
    mse_cat = mean_squared_error(y_test, y_pred_test)
    rmse_cat = np.sqrt(mse_cat)
    mae_cat = mean_absolute_error(y_test, y_pred_test)
    r2_cat = r2_score(y_test, y_pred_test)

    # Registrar métricas en MLflow
    mlflow.log_metric("mse", mse_cat)
    mlflow.log_metric("rmse", rmse_cat)
    mlflow.log_metric("mae", mae_cat)
    mlflow.log_metric("r2", r2_cat)

    print(f"--- RESULTADOS CATBOOST --- R2: {r2_cat:.4f} | MAE: {mae_cat:,.2f}")

    # 3. Inferencia, consolidación y formato final
    X_inf = df_inference.drop(columns=[col for col in cols_a_excluir if col in df_inference.columns])
    for col in cat_features:
        if col in X_inf.columns:
            X_inf[col] = X_inf[col].astype(str)
    df_inference['v_ref_2026_predicho'] = cat_model.predict(X_inf)

    X_sup_completo = df_supervised.drop(columns=[col for col in cols_a_excluir if col in df_supervised.columns])
    for col in cat_features:
        if col in X_sup_completo.columns:
            X_sup_completo[col] = X_sup_completo[col].astype(str)
    df_supervised['v_ref_2026_predicho'] = cat_model.predict(X_sup_completo)

    df_final_maestro = pd.concat([df_supervised, df_inference], ignore_index=True)
    df_final_maestro['hipervalorizacion_pct'] = ((df_final_maestro['v_ref_2026_predicho'] - df_final_maestro['v_ref_2025']) / df_final_maestro['v_ref_2025']) * 100
    df_final_maestro['tipo_registro'] = df_final_maestro['v_ref_2026'].apply(lambda x: 'Histórico Oficial' if pd.notnull(x) else 'Imputado / Predicho (Huérfano)')
    
    df_final_maestro['v_ref_2026_predicho'] = df_final_maestro['v_ref_2026_predicho'].round(0)
    df_final_maestro['hipervalorizacion_pct'] = df_final_maestro['hipervalorizacion_pct'].round(2)

    # 4. Registrar el modelo CatBoost en MLflow
    mlflow.catboost.log_model(cat_model, "catboost_model")

    # 5. Exportar archivos para Dash y API
    #gdf_export = gpd.GeoDataFrame(df_final_maestro, geometry=gpd.GeoSeries.from_wkt(df_final_maestro['geometry']), crs="EPSG:4326")
    #output_gpkg = "/dbfs/FileStore/output_hipervalorizacion_bogota_catboost.gpkg" # Ruta persistente en Databricks
    #gdf_export.to_file(output_gpkg, driver="GPKG")
    
    #print(f"\n¡Experimento registrado en MLflow y archivo .gpkg guardado en: {output_gpkg}!")